# 04 – Phân loại khả năng khách hàng mua lại

Bài toán: dự đoán `target_repeat_90d` – khách hàng có quay lại mua trong cửa sổ dự báo sau mốc cutoff (2011-09-01) hay không.

- Đặc trưng chỉ tổng hợp từ giao dịch **trước** cutoff (không rò rỉ dữ liệu theo thời gian).
- Chia train/test **trước** khi fit bất kỳ bộ biến đổi nào; ngưỡng phân loại chọn trên validation tách từ train, tập test chỉ đánh giá một lần.
- Yêu cầu: đã chạy `src/pipeline.py` (hoặc notebook 01) để có `data/processed/retail_processed.parquet`.

## 0. Chuẩn bị tập dữ liệu mô hình hóa

In [ ]:
from pathlib import Path
from src.classification_features import run_feature_engineering

PROCESSED = Path("../data/processed")
MODEL_PATH = PROCESSED / "customer_modeling_dataset.parquet"

# Sinh tập dữ liệu mô hình hóa nếu chưa có (cần chạy notebook 01 / src.pipeline trước để có retail_processed.parquet)
if not MODEL_PATH.exists():
    run_feature_engineering(
        processed_path=str(PROCESSED / "retail_processed.parquet"),
        output_path=str(MODEL_PATH),
        cutoff_date="2011-09-01",
    )
else:
    print(f"Đã có {MODEL_PATH}, bỏ qua bước tạo dữ liệu.")

## 1. Nạp dữ liệu và chia train/test

In [ ]:
import os
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from src.pipeline import build_sklearn_preprocessor
from src.classification_features import FEATURE_WHITELIST

df_model = pd.read_parquet("../data/processed/customer_modeling_dataset.parquet")
print(f"Tổng số khách hàng huấn luyện/kiểm thử: {len(df_model):,}")
print("Phân phối biến mục tiêu (target_repeat_90d):")
print(df_model["target_repeat_90d"].value_counts(normalize=True).round(4))

feature_cols = (
    FEATURE_WHITELIST["log_num_cols"]
    + FEATURE_WHITELIST["linear_num_cols"]
    + FEATURE_WHITELIST["cat_cols"]
)
X = df_model[feature_cols].copy()
y = df_model["target_repeat_90d"].values

# Chia tập dữ liệu trước khi fit bất kỳ bộ biến đổi nào (chống rò rỉ dữ liệu)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} mẫu | Test: {len(X_test):,} mẫu")

## 2. So sánh mô hình bằng Stratified 5-fold CV (chỉ trên tập train)

In [ ]:
models = {
    "Dummy Baseline (Most Frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression (Balanced)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "k-NN (k=15, Distance)": KNeighborsClassifier(n_neighbors=15, weights="distance"),
    "Gaussian Naive Bayes": GaussianNB(),
    "Random Forest (Balanced)": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200, max_depth=5, learning_rate=0.05, random_state=42
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

cv_results = []
for name, clf in models.items():
    prep = build_sklearn_preprocessor(
        log_num_cols=FEATURE_WHITELIST["log_num_cols"],
        linear_num_cols=FEATURE_WHITELIST["linear_num_cols"],
        cat_cols=FEATURE_WHITELIST["cat_cols"],
    )
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results.append({
        "Model": name,
        "ROC-AUC (CV)": f"{scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}",
        "PR-AUC (CV)": f"{scores['test_pr_auc'].mean():.4f} ± {scores['test_pr_auc'].std():.4f}",
        "F1-Score (CV)": f"{scores['test_f1'].mean():.4f}",
        "Precision (CV)": f"{scores['test_precision'].mean():.4f}",
        "Recall (CV)": f"{scores['test_recall'].mean():.4f}",
        "_roc_mean": scores["test_roc_auc"].mean(),
    })

results_df = pd.DataFrame(cv_results).sort_values("_roc_mean", ascending=False).drop(columns=["_roc_mean"])
display(results_df)

## 3. Chọn ngưỡng xác suất tối ưu F1 và đánh giá trên tập test

In [ ]:
# Tách một phần nhỏ từ X_train làm validation để chọn ngưỡng tối ưu F1 mà không chạm vào X_test
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

best_prep = build_sklearn_preprocessor(
    log_num_cols=FEATURE_WHITELIST["log_num_cols"],
    linear_num_cols=FEATURE_WHITELIST["linear_num_cols"],
    cat_cols=FEATURE_WHITELIST["cat_cols"],
)
best_pipe = Pipeline([
    ("prep", best_prep),
    ("clf", HistGradientBoostingClassifier(max_iter=200, max_depth=5, learning_rate=0.05, random_state=42)),
])
best_pipe.fit(X_tr, y_tr)

val_probs = best_pipe.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx])
print(f"Ngưỡng xác suất tối ưu F1 trên tập Validation: {optimal_threshold:.3f} (F1_val = {f1_scores[best_idx]:.4f})")

# Huấn luyện lại trên toàn bộ X_train và đánh giá duy nhất một lần trên X_test
best_pipe.fit(X_train, y_train)
test_probs = best_pipe.predict_proba(X_test)[:, 1]
test_preds_default = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= optimal_threshold).astype(int)

print(f"\n=== KẾT QUẢ TRÊN TẬP TEST (ROC-AUC = {roc_auc_score(y_test, test_probs):.4f} | PR-AUC = {average_precision_score(y_test, test_probs):.4f}) ===")
print("\n--- Ngưỡng mặc định (0.50) ---")
print(classification_report(y_test, test_preds_default, digits=4))
print(f"--- Ngưỡng tối ưu ({optimal_threshold:.3f}) ---")
print(classification_report(y_test, test_preds_tuned, digits=4))

## 4. Trực quan hóa: ROC, Precision-Recall, Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_probs)
axes[0].plot(fpr, tpr, color="#1f77b4", lw=2, label=f"HistGB (AUC = {roc_auc_score(y_test, test_probs):.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random (AUC = 0.500)")
axes[0].set_title("Đường cong ROC (Tập Test)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].legend(loc="lower right")

# 2. Precision-Recall Curve
prec_t, rec_t, _ = precision_recall_curve(y_test, test_probs)
axes[1].plot(rec_t, prec_t, color="#2ca02c", lw=2, label=f"HistGB (PR-AUC = {average_precision_score(y_test, test_probs):.3f})")
axes[1].axhline(y_test.mean(), color="gray", linestyle="--", label=f"Baseline ({y_test.mean():.3f})")
axes[1].set_title("Đường cong Precision-Recall (Tập Test)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

# 3. Confusion Matrix tại ngưỡng tối ưu
cm = confusion_matrix(y_test, test_preds_tuned)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Không mua lại (0)", "Mua lại 90d (1)"])
disp.plot(ax=axes[2], cmap="Blues", colorbar=False)
axes[2].set_title(f"Ma trận nhầm lẫn (Ngưỡng = {optimal_threshold:.2f})")

plt.tight_layout()
os.makedirs("../reports/figures", exist_ok=True)
fig.savefig("../reports/figures/05_classification_evaluation.png", dpi=300, bbox_inches="tight")
plt.show()